In [73]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import matplotlib as mpl
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

BLUE = "#5EDDF3"
ORANGE = "#F78441"
GRAY = "#8E9196"
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False
mpl.rcParams["font.size"] = 10.5

from pathlib import Path
RAW_PATH = next(path for path in (Path('data/acdc_dataset.csv'), Path('../data/acdc_dataset.csv')) if path.exists())
df_raw = pd.read_csv(RAW_PATH, sep=";")
print(df_raw.shape)
df_raw.head()

(15465, 31)


,member_id,age,gender,industry,country,is_team_account,n_colleagues_on_account,plan_tier,plan_days_per_week,membership_start,membership_end,last_badge_swipe,days_since_last_visit,total_visits,avg_weekly_visits,entitled_visits_last_period,actual_visits_last_period,meeting_room_hours,event_space_hours,guest_passes_issued,uses_meeting_rooms,uses_event_space,uses_guest_passes,uses_print,uses_mail_handling,uses_phone_booths,uses_day_passes,uses_storage,uses_bike_locker,n_renewals,lifetime_value
0,CW000001,38.0,Female,Information Technology,Portugal,False,0,flex,3,2021-08-01,2023-03-06,2023-03-23,-17,220,2.65,26.09,22,0.0,0.0,18.0,False,False,True,0,False,True,True,False,False,2,4477.97
1,CW000002,38.0,Male,Information Technology,Portugal,True,2,HOT_DESK,2,2023-11-24,1970-01-01,2023-11-24,37,2,0.32,10.68,1,0.0,0.0,0.0,False,False,False,0,True,False,False,False,False,0,165.86
2,CW000003,43.0,Male,Education,DE,True,1,flex,3,2021-12-03,1970-01-01,2023-07-22,163,19,0.17,26.09,2,0.0,0.0,0.0,False,False,False,0,False,False,False,False,False,1,4486.17
3,CW000004,38.0,F,Legal,portugal,False,0,flex,3,2022-01-02,2022-02-15,2022-01-02,44,8,1.23,18.74,9,0.0,0.0,0.0,False,False,False,0,False,False,False,False,False,0,258.63
4,CW000005,45.0,Female,Information Technology,United Kingdom,False,0,hot_desk,2,2023-10-24,1970-01-01,2023-12-17,15,4,0.38,17.39,3,0.0,0.0,0.0,False,False,False,0,False,False,True,False,False,0,335.74


In [30]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15465 entries, 0 to 15464
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   member_id                    15465 non-null  object 
 1   age                          14907 non-null  float64
 2   gender                       14935 non-null  object 
 3   industry                     15465 non-null  object 
 4   country                      15035 non-null  object 
 5   is_team_account              15465 non-null  bool   
 6   n_colleagues_on_account      15465 non-null  int64  
 7   plan_tier                    15360 non-null  object 
 8   plan_days_per_week           15465 non-null  int64  
 9   membership_start             15465 non-null  object 
 10  membership_end               15465 non-null  object 
 11  last_badge_swipe             15465 non-null  object 
 12  days_since_last_visit        15465 non-null  int64  
 13  total_visits    

In [31]:
df_raw.dtypes

member_id                       object
age                            float64
gender                          object
industry                        object
country                         object
is_team_account                   bool
n_colleagues_on_account          int64
plan_tier                       object
plan_days_per_week               int64
membership_start                object
membership_end                  object
last_badge_swipe                object
days_since_last_visit            int64
total_visits                     int64
avg_weekly_visits              float64
entitled_visits_last_period    float64
actual_visits_last_period        int64
meeting_room_hours             float64
event_space_hours              float64
guest_passes_issued            float64
uses_meeting_rooms                bool
uses_event_space                  bool
uses_guest_passes                 bool
uses_print                       int64
uses_mail_handling                bool
uses_phone_booths        

In [35]:
round(df_raw.isna().sum()/len(df_raw)*100,2)

member_id                      0.00
age                            3.61
gender                         3.43
industry                       0.00
country                        2.78
is_team_account                0.00
n_colleagues_on_account        0.00
plan_tier                      0.68
plan_days_per_week             0.00
membership_start               0.00
membership_end                 0.00
last_badge_swipe               0.00
days_since_last_visit          0.00
total_visits                   0.00
avg_weekly_visits              0.00
entitled_visits_last_period    0.00
actual_visits_last_period      0.00
meeting_room_hours             8.39
event_space_hours              0.00
guest_passes_issued            8.22
uses_meeting_rooms             0.00
uses_event_space               0.00
uses_guest_passes              0.00
uses_print                     0.00
uses_mail_handling             0.00
uses_phone_booths              0.00
uses_day_passes                0.00
uses_storage                

In [38]:
print(df_raw.duplicated().sum())
print(df_raw["member_id"].duplicated().sum())

15
15


In [44]:
def cramers_v(confusion_matrix: pd.DataFrame) -> float:
    """Bias-corrected Cramer's V effect size for a chi-square test of association."""
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)
    return float(np.sqrt(phi2_corr / min(k_corr - 1, r_corr - 1)))


def test_missingness_mechanism(df: pd.DataFrame, target_col: str, candidate_predictors: list) -> pd.DataFrame:
    """
    For `target_col`'s missing-value indicator, test association against each
    column in `candidate_predictors` via chi-square + Cramer's V.
    Returns one row per predictor, sorted by association strength (strongest first).
    """
    indicator = df[target_col].isna()
    rows = []
    for predictor in candidate_predictors:
        if predictor == target_col:
            continue
        sub = pd.DataFrame({"missing": indicator, "predictor": df[predictor]}).dropna(subset=["predictor"])
        if sub["predictor"].nunique() < 2 or sub["missing"].nunique() < 2:
            continue
        table = pd.crosstab(sub["missing"], sub["predictor"])
        chi2, p, _, _ = chi2_contingency(table)
        v = cramers_v(table)
        rows.append({"predictor": predictor, "cramers_v": round(v, 3), "p_value": p, "n": len(sub)})
    result = pd.DataFrame(rows).sort_values("cramers_v", ascending=False).reset_index(drop=True)
    return result


def verdict_from_max_v(max_v: float) -> str:
    if max_v >= 0.2:
        return "MAR / MNAR (possible pattern)"
    elif max_v >= 0.1:
        return "borderline -- worth a closer look"
    else:
        return "MCAR (scattered, no pattern found)"


In [45]:
df_raw.columns

Index(['member_id', 'age', 'gender', 'industry', 'country', 'is_team_account', 'n_colleagues_on_account', 'plan_tier', 'plan_days_per_week', 'membership_start', 'membership_end', 'last_badge_swipe', 'days_since_last_visit', 'total_visits',
       'avg_weekly_visits', 'entitled_visits_last_period', 'actual_visits_last_period', 'meeting_room_hours', 'event_space_hours', 'guest_passes_issued', 'uses_meeting_rooms', 'uses_event_space', 'uses_guest_passes', 'uses_print',
       'uses_mail_handling', 'uses_phone_booths', 'uses_day_passes', 'uses_storage', 'uses_bike_locker', 'n_renewals', 'lifetime_value'],
      dtype='object')

In [46]:
candidate_predictors = ['member_id', 'age', 'gender', 'industry', 'country', 'is_team_account', 'n_colleagues_on_account', 'plan_tier', 'plan_days_per_week', 'membership_start', 'membership_end', 'last_badge_swipe', 'days_since_last_visit', 'total_visits',
       'avg_weekly_visits', 'entitled_visits_last_period', 'actual_visits_last_period', 'meeting_room_hours', 'event_space_hours', 'guest_passes_issued', 'uses_meeting_rooms', 'uses_event_space', 'uses_guest_passes', 'uses_print',
       'uses_mail_handling', 'uses_phone_booths', 'uses_day_passes', 'uses_storage', 'uses_bike_locker', 'n_renewals', 'lifetime_value']
raw_missing_cols = [c for c in ['member_id', 'age', 'gender', 'industry', 'country', 'is_team_account', 'n_colleagues_on_account', 'plan_tier', 'plan_days_per_week', 'membership_start', 'membership_end', 'last_badge_swipe', 'days_since_last_visit', 'total_visits',
       'avg_weekly_visits', 'entitled_visits_last_period', 'actual_visits_last_period', 'meeting_room_hours', 'event_space_hours', 'guest_passes_issued', 'uses_meeting_rooms', 'uses_event_space', 'uses_guest_passes', 'uses_print',
       'uses_mail_handling', 'uses_phone_booths', 'uses_day_passes', 'uses_storage', 'uses_bike_locker', 'n_renewals', 'lifetime_value']
                     if df_raw[c].isna().any()]

first_pass_results = {}
for col in raw_missing_cols:
    preds = [p for p in candidate_predictors if p != col]
    first_pass_results[col] = test_missingness_mechanism(df_raw, col, preds)
    print(f"=== {col} (n missing = {df_raw[col].isna().sum()}) ===")
    print(first_pass_results[col])
    print()

=== age (n missing = 558) ===
                      predictor  cramers_v       p_value      n
0   entitled_visits_last_period      0.192  7.010567e-18  15465
1                  total_visits      0.084  4.765537e-03  15465
2             event_space_hours      0.080  8.943852e-03  15465
3              membership_start      0.062  1.449198e-01  15465
4         days_since_last_visit      0.061  2.733878e-02  15465
5             avg_weekly_visits      0.049  1.412238e-01  15465
6     actual_visits_last_period      0.034  6.099416e-02  15465
7                     member_id      0.031  4.622429e-01  15465
8                       country      0.027  9.950157e-02  15035
9            meeting_room_hours      0.024  4.425306e-01  14167
10                     industry      0.018  2.687362e-01  15465
11             uses_event_space      0.016  2.300429e-02  15465
12              is_team_account      0.013  6.237404e-02  15465
13                   uses_print      0.004  2.591959e-01  15465
14        

In [ ]:
first_pass_verdicts = pd.DataFrame([
    {
        "column": col,
        "strongest_predictor": res.iloc[0]["predictor"] if len(res) else None,
        "max_cramers_v": res.iloc[0]["cramers_v"] if len(res) else None,
        "verdict": verdict_from_max_v(res.iloc[0]["cramers_v"] if len(res) else 0.0),
    }
    for col, res in first_pass_results.items()
])
first_pass_verdicts

,column,strongest_predictor,max_cramers_v,verdict
0,age,entitled_visits_last_period,0.192,borderline -- worth a closer look
1,gender,guest_passes_issued,0.112,borderline -- worth a closer look
2,country,avg_weekly_visits,0.067,"MCAR (scattered, no pattern found)"
3,plan_tier,meeting_room_hours,0.397,MAR / MNAR (possible pattern)
4,meeting_room_hours,entitled_visits_last_period,0.184,borderline -- worth a closer look
5,guest_passes_issued,entitled_visits_last_period,0.167,borderline -- worth a closer look
6,lifetime_value,meeting_room_hours,0.280,MAR / MNAR (possible pattern)


In [77]:

df = df_raw.copy()


print("Negative ages:", (df["age"] < 0).sum())
print("Ages below 18:", (df["age"] < 18).sum())
print("Missing ages:", df["age"].isna().sum())
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

Negative ages: 0
Ages below 18: 98
Missing ages: 558
Minimum age: 1.0
Maximum age: 149.0


In [87]:

numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns

summary = df[numeric_columns].describe().T

print(summary)

                               count        mean         std    min    25%    50%     75%       max
age                          14907.0   34.858724   10.394649    1.0  29.00  34.00   40.00    149.00
n_colleagues_on_account      15465.0    0.537019    1.177668   -1.0   0.00   0.00    1.00      9.00
plan_days_per_week           15465.0    3.387262    1.458818    0.0   2.00   3.00    5.00      9.00
days_since_last_visit        15465.0  117.109279  275.532807 -956.0   4.00  13.00   55.00   1212.00
total_visits                 15465.0  110.406531  182.939340 -112.0   8.00  34.00  132.00   6530.00
avg_weekly_visits            15465.0    1.777214    1.937515    0.0   0.71   1.42    2.41     47.40
entitled_visits_last_period  15465.0   27.517393   13.480235    0.0  17.39  26.09   43.48     78.27
actual_visits_last_period    15465.0   13.800970   11.387207    0.0   5.00  10.00   20.00     60.00
meeting_room_hours           14167.0   33.075570  428.169678  -56.6   0.00   0.00    7.20  41220.00


In [91]:


def domain_rule_report(df: pd.DataFrame, feat: str, min_val: int, max_val: int) -> pd.DataFrame:
    rows = []
    # age: implausible if outside a plausible adult defendant range
    feat_num = pd.to_numeric(df[feat], errors="coerce")
    bad_val = df[feat].notna() & ~feat_num.between(min_val, max_val)
    rows.append({"column": feat, "rule": f"{min_val} <= {feat} <= {max_val}", "violations": int(bad_val.sum()),
                 "examples": sorted(feat_num[bad_val].unique().tolist())[:6]})


    return pd.DataFrame(rows)

for col in numeric_columns:
    print(f"=== {col} ===")
    print(domain_rule_report(df, col, 0, 100))
    print()



=== age ===
  column             rule  violations                                    examples
0    age  0 <= age <= 100          45  [131.0, 132.0, 134.0, 135.0, 136.0, 137.0]

=== n_colleagues_on_account ===
                    column                                 rule  violations examples
0  n_colleagues_on_account  0 <= n_colleagues_on_account <= 100         566     [-1]

=== plan_days_per_week ===
               column                            rule  violations examples
0  plan_days_per_week  0 <= plan_days_per_week <= 100           0       []

=== days_since_last_visit ===
                  column                               rule  violations                              examples
0  days_since_last_visit  0 <= days_since_last_visit <= 100        2691  [-956, -903, -805, -779, -734, -721]

=== total_visits ===
         column                      rule  violations                          examples
0  total_visits  0 <= total_visits <= 100        4950  [-112, -102, -88, -77, -75,

In [ ]:
df = pd.read_csv(RAW_PATH, sep=";")

df.head(5)

df.columns

df.info()

df.describe()

df.dtypes

In [ ]:
df.isnull().sum()

for col in df.columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))
    


member_id:
member_id
CW005845    2
CW009828    2
CW001390    2
CW006678    2
CW012056    2
           ..
CW005007    1
CW005008    1
CW005009    1
CW005010    1
CW015000    1
Name: count, Length: 15450, dtype: int64

age:
age
34.0     710
31.0     679
32.0     658
36.0     655
35.0     650
        ... 
143.0      1
139.0      1
91.0       1
149.0      1
147.0      1
Name: count, Length: 85, dtype: int64

gender:
gender
Female     7414
Male       6172
NaN         530
Other       465
female      169
F           143
 Female     141
M           138
male        132
 Male       129
other        15
 Other        9
O             8
Name: count, dtype: int64

industry:
industry
Information Technology    3322
Consulting                1896
Marketing & PR            1835
Design & Creative         1822
Finance                   1134
Other                     1100
Legal                      809
Education                  727
Unknown                    640
Health                     571
Real Estate 

In [27]:
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].unique())


member_id:
['CW000001' 'CW000002' 'CW000003' ... 'CW014998' 'CW014999' 'CW015000']

age:
[ 38.  43.  45.  39.  23.  33.  30.  44.  37. 131.  20.  40.  32.  21.
  29.  25.  28.  51.  41.  nan  31.  42.  34.  36.  50.  24.  19.  47.
  54.  26.  53.  27.  46.  18.  48.  63.  35.  22.  59.  49.   4.   3.
  58.  72.   9.   8.  57.  52.  73. 132.  60.  55. 144.  61.   1.   6.
 141.   2.   7. 137.  66. 147. 140. 142.  56.  62. 145.   5.  65.  83.
  14. 138. 136. 146. 134.  70.  74.  93.  92. 143. 139.  91.  64. 149.
 135.]

gender:
['Female' 'Male' 'F' 'Other' nan ' Female' 'M' 'male' ' Male' 'other'
 ' Other' 'female' 'O']

industry:
['Information Technology' 'Education' 'Legal' 'Health' 'Unknown' 'Other'
 'Consulting' 'Design & Creative' 'information technology' 'Finance'
 'Marketing & PR' 'EDUCATION' 'DESIGN & CREATIVE' 'Marketing and PR'
 'Real Estate' 'other' 'consulting' 'MARKETING & PR'
 'Information technology' 'design & creative' 'LEGAL' 'marketing & pr'
 'INFORMATION TECHNOLOGY' 'h